# Solar Filament Segmentation Challenge 2026 -- ResNet50 fine-tuning (Kaggle GPU)

Clones the `jp-finetune-resnet50` branch and drives the same `src/train.py` /
`src/infer.py` / `src/submission.py` used locally -- logic lives in one place (the
repo), not duplicated into notebook cells. This is Stage 4 of `RESNET_PRETRAIN_PLAN.md`:
supervised fine-tuning of the GONG Halpha BYOL-pretrained ResNet50 encoder on the
labeled MAGFiLO filament data, using `jp-mvp1`'s existing semantic-mask +
connected-components segmentation head (unchanged) -- only the encoder and the
fine-tuning recipe (loss, layer-wise LR decay, freeze warmup, resolution,
model selection on PQ) are new. See `FINETUNE_PLAN.md` for the full design and the
decisions behind it.

### 0. Clone the repo

Requires the `jp-finetune-resnet50` branch to already be pushed to `origin` -- if
this clone fails with "Remote branch jp-finetune-resnet50 not found", push it from
local first.

In [ ]:
!git clone -b jp-finetune-resnet50 https://github.com/jprakash-1/Solar-Filament-Segmentation.git


In [ ]:
%cd Solar-Filament-Segmentation
!ls


In [ ]:
# re-run-safe: pick up any commits pushed after the kernel started
!git pull origin jp-finetune-resnet50


### 1. Install dependencies + confirm GPU

**Do not** `pip install torch torchvision` here -- Kaggle's base image ships a
PyTorch build already matched to whatever GPU this kernel was assigned. Reinstalling
from PyPI pulls the newest wheel, which can drop kernel support for older cards
(`jp-mvp1`'s README documents hitting exactly this on a P100). Install everything
**except** torch/torchvision from requirements.txt.

In [ ]:
!grep -v -E "^(torch|torchvision)\b" requirements.txt > /tmp/requirements_kaggle.txt
!pip install -q -r /tmp/requirements_kaggle.txt

In [ ]:
import torch
print("torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("gpu count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  device {i}:", torch.cuda.get_device_name(i))


### 2. Locate the mounted data + the BYOL encoder checkpoint

Auto-detects both inputs under `/kaggle/input` by globbing, same pattern
`train_mvp1_kaggle.ipynb` already uses for the competition data -- neither the
competition-data mount folder name nor the encoder-checkpoint dataset's exact slug
(not yet decided, see `FINETUNE_PLAN.md`) is hardcoded here.

In [ ]:
!find /kaggle/input -maxdepth 4 2>/dev/null || echo "(not running on Kaggle -- /kaggle/input doesn't exist)"


In [ ]:
import glob
from pathlib import Path

if Path("/kaggle/input").exists():
    matches = glob.glob("/kaggle/input/**/MAGFiLO_1.0_Annotations_kaggle2026_train.json", recursive=True)
    assert matches, "Could not find the training annotation json under /kaggle/input -- check the attached dataset and adjust this cell."
    DATA_JSON = Path(matches[0])
    TRAIN_IMAGES_DIR = DATA_JSON.parent / "train_images"
    TEST_IMAGES_DIR = DATA_JSON.parent.parent / "test" / "test_images"
else:
    DATA_JSON = Path("data/raw/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json")
    TRAIN_IMAGES_DIR = Path("data/raw/MAGFiLO_1.0_Kaggle_2026/train/train_images")
    TEST_IMAGES_DIR = Path("data/raw/MAGFiLO_1.0_Kaggle_2026/test/test_images")

print("DATA_JSON:", DATA_JSON, "exists:", DATA_JSON.exists())
print("TRAIN_IMAGES_DIR:", TRAIN_IMAGES_DIR, "exists:", TRAIN_IMAGES_DIR.exists())
print("TEST_IMAGES_DIR:", TEST_IMAGES_DIR, "exists:", TEST_IMAGES_DIR.exists())
assert DATA_JSON.exists() and TRAIN_IMAGES_DIR.exists() and TEST_IMAGES_DIR.exists(), \
    "adjust the path-detection logic above to match this dataset's actual mount layout"


In [ ]:
import glob
from pathlib import Path

# Exported by scripts/pretrain_resnet/export_encoder.py on jp-pretraining-data-prep,
# attached here as its own Kaggle Dataset input -- slug not fixed by this notebook,
# just globbed for by filename.
if Path("/kaggle/input").exists():
    ckpt_matches = [
        p for p in glob.glob("/kaggle/input/**/*.pt", recursive=True)
        if "encoder" in Path(p).name.lower()
    ]
    assert ckpt_matches, (
        "Could not find an exported encoder checkpoint (*.pt with 'encoder' in the "
        "filename) under /kaggle/input -- attach the resnet50 BYOL encoder checkpoint "
        "dataset (see scripts/pretrain_resnet/export_encoder.py on jp-pretraining-data-prep), "
        "or fall back to ENCODER_CHECKPOINT = None below to fine-tune from plain ImageNet "
        "weights instead (still a valid, if untested-here, comparison point)."
    )
    ENCODER_CHECKPOINT = Path(ckpt_matches[0])
else:
    ENCODER_CHECKPOINT = None

print("ENCODER_CHECKPOINT:", ENCODER_CHECKPOINT)


### 3. Verify the checkpoint loads before spending GPU time on it

One-time structural + real-weight check (see `src/model.py`'s module docstring and
`scripts/verify_encoder_checkpoint.py` for what this actually confirms and why it's
needed -- smp's own ImageNet stem adaptation is NOT numerically compatible with the
BYOL pretraining's stem, a real mismatch already measured on the pretraining branch;
this check instead confirms the *checkpoint-loading* path, which sidesteps that
entirely, actually lines up key-for-key).

In [ ]:
!python scripts/verify_encoder_checkpoint.py --checkpoint "{ENCODER_CHECKPOINT}"


### 3.5. Resume from a prior session, if any

Kaggle kernels don't persist `/kaggle/working` between sessions -- multi-session
training (this run is `--epochs 30`, likely more than one ~9-12h session) means
mounting the *previous* session's `finetune_resnet50_latest.pt` (see "Outputs"
at the bottom -- version it as its own Kaggle Dataset after each session, same
`RESNET_PRETRAIN_PLAN.md` section 9 pattern the BYOL pretraining notebook uses)
as an input here and resuming from it. Auto-detected by filename, same as the
encoder checkpoint above; absent on a first session, which is expected.

In [ ]:
if Path("/kaggle/input").exists():
    resume_matches = glob.glob("/kaggle/input/**/finetune_resnet50_latest.pt", recursive=True)
    RESUME_FROM = Path(resume_matches[0]) if resume_matches else None
else:
    RESUME_FROM = None

RESUME_FLAG = f'--resume "{RESUME_FROM}"' if RESUME_FROM is not None else ""
print("RESUME_FROM:", RESUME_FROM if RESUME_FROM else "(none found -- starting fresh)")


### 4. Train

Key differences from `train_mvp1_kaggle.ipynb`'s plain resnet18/ImageNet run:
`--encoder-name resnet50 --encoder-checkpoint` loads the domain-pretrained encoder
(see `src/model.py`); `--img-size 2048` trains at full native resolution (no
downsampling, per explicit project direction -- see `FINETUNE_PLAN.md` for the
BatchNorm-calibration tradeoff this accepts); `--loss tversky_bce` is the
recall-biased fine-tuning loss `PRETRAIN_PLAN.md` section 5.4 specifies;
`--freeze-epochs 3` keeps the encoder frozen for the first 3 epochs (forgetting
guard, section 5.6); `--head-lr`/`--encoder-lr-decay` set up the layer-wise LR
schedule (section 5.5). The resume flag from section 3.5 is empty on a first session
and resumes model/optimizer/scaler/epoch/best-val-PQ state on later ones --
`--epochs 30` is the TOTAL across all sessions, not per-session, so a resumed run
trains only the remaining epochs.

**`--batch-size 4` is a starting point, not a measured one** -- a ResNet50 U-Net at
2048x2048 is far heavier than `jp-mvp1`'s 256px (batch 64) or even its one 1024px
run (batch 8). Re-tune empirically on the real hardware this actually runs on
(`kaggle.md`'s own explicit "re-tune batch size every time you change resolution"
guidance) -- raise it if a first run shows large unused GPU memory. **If it OOMs,
reach for `--grad-accum-steps` before lowering `--batch-size`** -- e.g.
`--batch-size 2 --grad-accum-steps 2` recovers the same effective batch size at
batch-size-2's peak memory (`src/train.py`'s `run_train_epoch` docstring has the
details, including how it's handled under DDP). `--amp auto` enables mixed
precision automatically on CUDA, which helps materially at this resolution.
`--early-stopping-patience 5` stops the run if val PQ hasn't improved in 5 epochs,
so a stalled/overfitting run doesn't silently burn the rest of the epoch budget.

Same DDP setup as `train_mvp1_kaggle.ipynb` otherwise (`torchrun`, `NCCL_P2P_DISABLE=1`,
`device_id`-explicit `init_process_group`, rank-0-only validation through the
unwrapped module) -- all inherited unchanged from `jp-mvp1`'s `src/distributed.py`
and the gotchas documented in its README, not re-derived here. One DDP-specific fix
this branch needed that `jp-mvp1` didn't: `find_unused_parameters=True` on the DDP
wrapper, required because the freeze-warmup schedule changes which parameters get
gradients epoch to epoch -- see `src/train.py`'s comment at the `DDP(...)` call for
the real crash this fixes (hit on an actual T4 x2 run before this fix landed).

In [ ]:
!NCCL_P2P_DISABLE=1 torchrun --nproc_per_node=$(python -c "import torch; print(max(1, torch.cuda.device_count()))") -m src.train \
    --data-json "{DATA_JSON}" \
    --images-dir "{TRAIN_IMAGES_DIR}" \
    --encoder-name resnet50 \
    --encoder-checkpoint "{ENCODER_CHECKPOINT}" \
    --img-size 2048 \
    --loss tversky_bce \
    --freeze-epochs 3 \
    --head-lr 1e-3 \
    --encoder-lr-decay 0.75 \
    --epochs 30 \
    --batch-size 4 \
    --early-stopping-patience 5 \
    --num-workers 4 \
    --amp auto \
    --checkpoint-out outputs/checkpoints/finetune_resnet50_kaggle.pt \
    --latest-checkpoint-out outputs/checkpoints/finetune_resnet50_latest.pt \
    --log-csv outputs/logs/finetune_log_kaggle.csv \
    {RESUME_FLAG}


### 5. Training curve (loss, Dice, and val PQ)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log = pd.read_csv("outputs/logs/finetune_log_kaggle.csv")
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(log["epoch"], log["train_loss"], label="train_loss")
axes[0].plot(log["epoch"], log["val_loss"], label="val_loss")
axes[0].set_xlabel("epoch"); axes[0].legend(); axes[0].set_title("Loss")
axes[1].plot(log["epoch"], log["train_dice"], label="train_dice")
axes[1].plot(log["epoch"], log["val_dice"], label="val_dice")
axes[1].set_xlabel("epoch"); axes[1].legend(); axes[1].set_title("Dice")
axes[2].plot(log["epoch"], log["val_pq_mean"], label="val_pq_mean")
axes[2].plot(log["epoch"], log["val_pq_pooled"], label="val_pq_pooled")
axes[2].set_xlabel("epoch"); axes[2].legend(); axes[2].set_title("Val Panoptic Quality")
fig.tight_layout()
plt.show()


### 6. Local validation (Panoptic Quality + Dice)

Runs the full inference + postprocessing pipeline against the held-out
`file_name`-grouped val split -- the same instance-splitting logic used for the real
submission, and the authoritative native-resolution PQ number (training's own
per-epoch val PQ, section 5, is computed at `--img-size` resolution for speed --
identical to this at `img_size=2048`, an approximation at any smaller resolution).

In [ ]:
!python -m src.infer \
    --checkpoint outputs/checkpoints/finetune_resnet50_kaggle.pt \
    --split val \
    --data-json "{DATA_JSON}" \
    --train-images-dir "{TRAIN_IMAGES_DIR}" \
    --device cuda


### 7. Inference -> submission.csv

In [ ]:
!python -m src.infer \
    --checkpoint outputs/checkpoints/finetune_resnet50_kaggle.pt \
    --split test \
    --test-images-dir "{TEST_IMAGES_DIR}" \
    --out outputs/submissions/finetune_resnet50_kaggle.csv \
    --device cuda


In [ ]:
!python -m src.submission --validate outputs/submissions/finetune_resnet50_kaggle.csv --test-images-dir "{TEST_IMAGES_DIR}"


### 8. Stage for one-click Kaggle submission

In [ ]:
import shutil

shutil.copy("outputs/submissions/finetune_resnet50_kaggle.csv", "/kaggle/working/submission.csv")
pd.read_csv("/kaggle/working/submission.csv").head()


### Outputs

- `outputs/checkpoints/finetune_resnet50_kaggle.pt` -- best-val-PQ model weights
  (encoder + decoder), plus `encoder_name`/`img_size`/`loss`/`val_pq` metadata
  `src/infer.py` reads back automatically. This is the one to submit from.
- `outputs/checkpoints/finetune_resnet50_latest.pt` -- saved every epoch
  unconditionally (model + optimizer + scaler + schedule state). **If this run
  didn't finish all 30 epochs** (session time limit, early stopping aside),
  version this file as a new Kaggle Dataset and mount it as input on the next
  session -- section 3.5 above auto-detects it and resumes automatically.
- `outputs/logs/finetune_log_kaggle.csv` -- per-epoch train/val loss + Dice + PQ
  (only from `start_epoch` onward on a resumed run, not the full history)
- `outputs/submissions/finetune_resnet50_kaggle.csv` / `/kaggle/working/submission.csv`
  -- ready to submit
- **The actual ablation this pipeline exists to answer** (`RESNET_PRETRAIN_PLAN.md`
  section 10): rerun section 4 with `ENCODER_CHECKPOINT` forced to `None` (plain
  ImageNet-init resnet50) and again with `--encoder-name resnet18 --encoder-checkpoint`
  omitted (the exact `jp-mvp1` baseline recipe) to get the three comparable val PQ
  numbers -- everything else (head, loss, resolution, schedule) held fixed per
  `kaggle.md`'s "change one variable at a time" discipline.